# Capítulo 9: Desafíos y Buenas Prácticas

## 9.1. Desafíos en la implementación

### Cuadro 9.1: Principales desafíos y estrategias de mitigación

| Desafío | Descripción | Mitigación |
| :--- | :--- | :--- |
| **Calidad de los datos** | Los modelos dependen de datos precisos y representativos. Datos incompletos generan falsos positivos/negativos. | Pipelines de calidad de datos; validación automática. |
| **Interpretabilidad** | Modelos de DL son "cajas negras"; dificultan auditorías y la confianza del analista. | SHAP, LIME; preferir modelos interpretables cuando sea posible. |
| **Sesgo en datos** | Datos de entrenamiento sesgados producen modelos que discriminan ciertos patrones. | Análisis de sesgo; SMOTE; revisión periódica de datasets. |
| **Ataques adversariales** | Atacantes pueden manipular entradas para evadir el modelo de IA. | Entrenamiento adversarial; monitoreo de distribución de entradas. |
| **Privacidad de datos** | Los logs pueden contener información personal sensible (GDPR, CCPA). | Anonimización; aprendizaje federado; cifrado de datos. |

---
## 9.2. Buenas prácticas de desarrollo seguro

### Lista de verificación de buenas prácticas

1. **Controlar versiones del modelo:** usar MLflow o DVC para registrar experimentos y versiones.
2. **Separar entornos:** no usar el mismo entorno para entrenamiento y producción.
3. **Validar entradas:** sanitizar y validar datos que entran al modelo en producción para prevenir inyección de datos.
4. **Monitorear deriva de datos (data drift):** usar herramientas como Evidently AI para detectar cambios en la distribución de datos.
5. **Reentrenar periódicamente:** el panorama de amenazas cambia; los modelos deben actualizarse con nuevas muestras.
6. **Aplicar XAI:** documentar las razones de cada alerta para facilitar auditorías y reducir fatiga del analista.
7. **Gestionar secretos de forma segura:** nunca incrustar tokens o contraseñas en el código; usar variables de entorno o bóvedas (Vault).

---
## 9.3. Monitoreo de deriva de datos en producción

La **deriva de datos** (data drift) ocurre cuando la distribución de los datos en producción cambia respecto a los datos de entrenamiento, degradando el rendimiento del modelo.

In [ ]:
# Listing 9.1: Detección de deriva de datos con estadístico KS

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp

def detectar_deriva(datos_referencia: pd.DataFrame,
                    datos_produccion: pd.DataFrame,
                    umbral_p: float = 0.05) -> dict:
    """
    Compara distribuciones de producción con los datos de referencia
    usando el test de Kolmogorov-Smirnov.
    Retorna un diccionario con columnas afectadas.
    """
    resultados = {}
    for columna in datos_referencia.columns:
        stat, p_valor = ks_2samp(
            datos_referencia[columna].dropna(),
            datos_produccion[columna].dropna()
        )
        hay_deriva = p_valor < umbral_p
        resultados[columna] = {
            'statistic': round(stat, 4),
            'p_value':   round(p_valor, 6),
            'deriva':    hay_deriva
        }

    columnas_con_deriva = [
        c for c, r in resultados.items() if r['deriva']
    ]
    if columnas_con_deriva:
        print(f"[ALERTA] Deriva detectada en: {columnas_con_deriva}")
    else:
        print("[OK] Sin deriva significativa en los datos de entrada.")

    return resultados

print("[OK] Función detectar_deriva() definida.")

### 9.3.1. Demostración de detección de deriva

Simulamos un escenario donde los datos de producción presentan un cambio de distribución respecto a los datos de referencia (entrenamiento).

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Datos de referencia: dataset original de tráfico de red
df_ref = pd.read_csv('data/network_traffic.csv').dropna()

# Codificar protocolo si es categórico
le = LabelEncoder()
if df_ref['protocol'].dtype == object:
    df_ref['protocol'] = le.fit_transform(df_ref['protocol'])

feature_cols = ['bytes_sent', 'bytes_recv', 'duration', 'src_port']

# Simular datos de producción con deriva
np.random.seed(99)
n_prod = 1000
df_prod = pd.DataFrame({
    'bytes_sent': np.random.exponential(1200, n_prod).astype(int),    # media mayor
    'bytes_recv': np.random.exponential(800, n_prod).astype(int),     # similar
    'duration':   np.random.exponential(60, n_prod).round(2),          # duración mayor
    'src_port':   np.random.randint(1024, 65535, n_prod),              # similar
})

print("=== Detección de deriva ===")
print("Referencia: datos de entrenamiento (network_traffic.csv)")
print("Producción: datos simulados con distribución alterada\n")

resultado_deriva = detectar_deriva(df_ref[feature_cols], df_prod[feature_cols])

print(f"\n{'Columna':<15} {'Statistic':>10} {'P-value':>12} {'Deriva':>8}")
print("-" * 47)
for col, res in resultado_deriva.items():
    marca = '⚠ SÍ' if res['deriva'] else '✓ NO'
    print(f"{col:<15} {res['statistic']:>10.4f} {res['p_value']:>12.6f} {marca:>8}")

In [ ]:
import matplotlib.pyplot as plt

# Visualización de distribuciones: referencia vs producción
fig, axes = plt.subplots(1, len(feature_cols), figsize=(18, 4))

for i, col in enumerate(feature_cols):
    axes[i].hist(df_ref[col], bins=40, alpha=0.5, color='steelblue', label='Referencia', density=True)
    axes[i].hist(df_prod[col], bins=40, alpha=0.5, color='coral', label='Producción', density=True)
    marca = '⚠' if resultado_deriva[col]['deriva'] else '✓'
    axes[i].set_title(f'{col} {marca}')
    axes[i].legend(fontsize=8)

plt.suptitle('Detección de Deriva: Referencia vs Producción', fontsize=13)
plt.tight_layout()
plt.savefig('data/data_drift_analysis.png', dpi=150)
plt.show()

---
---
# Capítulo 10: Integración Completa: Pipeline de Seguridad con IA

## 10.1. Arquitectura del sistema

```
Fuentes        →  Ingesta         →  Análisis IA        →  Decisión         →  Acción
─────────────     ──────────────     ─────────────────     ──────────────     ─────────────
Logs de red       Pipeline ETL       Anomaly Detection     Motor de reglas    Aislar sistema
Endpoints         Normalización      Malware Detection     Triaje SVM         Bloquear IP
SIEM              Feature Eng.       UBA                   Severidad          Notificar CSIRT
```

---
## 10.2. Pipeline completo en Python

In [ ]:
# Listing 10.1: Pipeline de seguridad integrado

import pandas as pd
import numpy as np
import joblib
import logging
import os
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.ensemble import IsolationForest, RandomForestClassifier

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s %(levelname)-8s %(message)s')

class PipelineSeguridad:
    """
    Pipeline integral de seguridad basado en IA.
    Integra detección de anomalías y clasificación de malware.
    """

    FEATURE_COLS = ['bytes_sent', 'bytes_recv',
                    'duration', 'src_port', 'dst_port']

    def __init__(self):
        self.scaler = MinMaxScaler()
        self.anomaly_det = IsolationForest(contamination=0.05,
                                          random_state=42,
                                          n_jobs=-1)
        self.malware_clf = RandomForestClassifier(n_estimators=200,
                                                 random_state=42,
                                                 n_jobs=-1)
        self.entrenado = False

    # Entrenamiento
    def entrenar(self,
                 df_trafico: pd.DataFrame,
                 df_malware: pd.DataFrame,
                 y_malware: pd.Series) -> None:
        """Entrena ambos modelos con datos históricos."""
        X_trafico = self.scaler.fit_transform(
            df_trafico[self.FEATURE_COLS]
        )
        self.anomaly_det.fit(X_trafico)
        logging.info("Isolation Forest entrenado.")

        self.malware_clf.fit(df_malware, y_malware)
        logging.info("Random Forest (malware) entrenado.")
        self.entrenado = True

    # Persistencia
    def guardar(self, ruta: str = 'models/pipeline_seguridad.pkl') -> None:
        os.makedirs(os.path.dirname(ruta), exist_ok=True)
        joblib.dump(self, ruta)
        logging.info(f"Pipeline guardado en {ruta}")

    @staticmethod
    def cargar(ruta: str = 'models/pipeline_seguridad.pkl') -> 'PipelineSeguridad':
        return joblib.load(ruta)

    # Inferencia
    def analizar_trafico(self, df: pd.DataFrame) -> pd.DataFrame:
        """Detecta anomalías en tráfico de red."""
        if not self.entrenado:
            raise RuntimeError("El pipeline no ha sido entrenado.")
        X = self.scaler.transform(df[self.FEATURE_COLS])
        df = df.copy()
        df['anomalia'] = self.anomaly_det.predict(X)
        df['score_anomalia'] = self.anomaly_det.decision_function(X)
        return df

    def analizar_archivo(self, caracteristicas: dict) -> dict:
        """Clasifica un archivo como benigno o malicioso."""
        if not self.entrenado:
            raise RuntimeError("El pipeline no ha sido entrenado.")
        X = pd.DataFrame([caracteristicas])
        pred = self.malware_clf.predict(X)[0]
        proba = self.malware_clf.predict_proba(X)[0]
        return {
            'clasificacion':  'malicioso' if pred == 1 else 'benigno',
            'prob_benigno':   round(proba[0], 4),
            'prob_malicioso': round(proba[1], 4),
        }

print("[OK] Clase PipelineSeguridad definida.")

### 10.2.1. Entrenamiento del pipeline integrado

In [ ]:
# Instanciar y entrenar el pipeline
pipeline = PipelineSeguridad()

# Cargar datos de tráfico de red (Capítulo 3)
df_trafico = pd.read_csv('data/network_traffic.csv').dropna()

# Codificar protocolo si es necesario
le = LabelEncoder()
if df_trafico['protocol'].dtype == object:
    df_trafico['protocol'] = le.fit_transform(df_trafico['protocol'])

# Cargar datos de malware (Capítulo 4)
df_malware = pd.read_csv('data/file_features.csv').dropna()
y_malware = df_malware.pop('label')

# Entrenar
pipeline.entrenar(df_trafico, df_malware, y_malware)

# Guardar pipeline completo
pipeline.guardar()

### 10.2.2. Análisis de tráfico en tiempo real

In [ ]:
# Simular tráfico "en tiempo real" con una mezcla de normal y anómalo
np.random.seed(77)
n_live = 500

df_live = pd.DataFrame({
    'bytes_sent': np.concatenate([
        np.random.exponential(500, n_live - 30).astype(int),
        np.random.exponential(8000, 30).astype(int)
    ]),
    'bytes_recv': np.concatenate([
        np.random.exponential(800, n_live - 30).astype(int),
        np.random.exponential(50, 30).astype(int)
    ]),
    'duration': np.concatenate([
        np.random.exponential(30, n_live - 30).round(2),
        np.random.exponential(300, 30).round(2)
    ]),
    'src_port': np.concatenate([
        np.random.randint(1024, 65535, n_live - 30),
        np.random.randint(1, 1024, 30)
    ]),
    'dst_port': np.concatenate([
        np.random.choice([80, 443, 53, 22], n_live - 30),
        np.random.choice([4444, 31337, 6667], 30)
    ]),
})

# Analizar tráfico
resultado = pipeline.analizar_trafico(df_live)
anomalias = resultado[resultado['anomalia'] == -1]

print(f"\n=== Análisis de tráfico en tiempo real ===")
print(f"Total de paquetes analizados: {len(resultado)}")
print(f"Anomalías detectadas:         {len(anomalias)}")
print(f"\nTop 5 anomalías (menor score = más anómalo):")
print(anomalias.sort_values('score_anomalia').head())

### 10.2.3. Análisis de archivos ejecutables

In [ ]:
# Analizar archivos sospechosos
archivos_test = [
    {
        'nombre': 'archivo_sospechoso.exe',
        'caracteristicas': {
            'entry_point': 512, 'image_base': 65536,
            'size_of_image': 15000, 'size_code_section': 2000,
            'dll_flag': 0, 'num_sections': 2,
            'entropia_max': 7.8, 'entropia_media': 7.2,
            'num_importaciones': 5, 'num_dlls_importadas': 2,
            'num_exportaciones': 0, 'file_size': 25000
        }
    },
    {
        'nombre': 'aplicacion_legitima.exe',
        'caracteristicas': {
            'entry_point': 16384, 'image_base': 4194304,
            'size_of_image': 250000, 'size_code_section': 100000,
            'dll_flag': 33120, 'num_sections': 5,
            'entropia_max': 5.2, 'entropia_media': 4.5,
            'num_importaciones': 180, 'num_dlls_importadas': 15,
            'num_exportaciones': 3, 'file_size': 850000
        }
    }
]

print("=== Análisis de archivos ===")
for archivo in archivos_test:
    res = pipeline.analizar_archivo(archivo['caracteristicas'])
    print(f"\n  Archivo: {archivo['nombre']}")
    print(f"  Clasificación: {res['clasificacion'].upper()}")
    print(f"  P(benigno):    {res['prob_benigno']:.4f}")
    print(f"  P(malicioso):  {res['prob_malicioso']:.4f}")

### 10.2.4. Verificación de deriva sobre datos en vivo

In [ ]:
# Aplicar detección de deriva comparando datos de entrenamiento vs datos en vivo
feature_cols_drift = ['bytes_sent', 'bytes_recv', 'duration', 'src_port']

print("=== Verificación de deriva: Entrenamiento vs Datos en vivo ===")
resultado_drift = detectar_deriva(
    df_trafico[feature_cols_drift],
    df_live[feature_cols_drift]
)

print(f"\n{'Columna':<15} {'Statistic':>10} {'P-value':>12} {'Deriva':>8}")
print("-" * 47)
for col, res in resultado_drift.items():
    marca = '⚠ SÍ' if res['deriva'] else '✓ NO'
    print(f"{col:<15} {res['statistic']:>10.4f} {res['p_value']:>12.6f} {marca:>8}")

---
## 10.3. Resumen final del pipeline

### Artefactos generados en todo el proyecto

In [ ]:
import os

print("=" * 60)
print("  PIPELINE DE CIBERSEGURIDAD CON IA - RESUMEN FINAL")
print("=" * 60)

# Listar modelos
print("\n📁 models/")
if os.path.exists('models'):
    for f in sorted(os.listdir('models')):
        size = os.path.getsize(os.path.join('models', f))
        print(f"   {f:<40} ({size:>10,} bytes)")

# Listar datos
print("\n📁 data/")
if os.path.exists('data'):
    for f in sorted(os.listdir('data')):
        size = os.path.getsize(os.path.join('data', f))
        print(f"   {f:<40} ({size:>10,} bytes)")

print("\n" + "=" * 60)
print("  Notebooks ejecutados:")
print("  01. Configuración del Entorno       (Cap. 2)")
print("  02. Detección de Amenazas            (Cap. 3)")
print("  03. Detección de Malware             (Cap. 4)")
print("  04. Respuesta a Incidentes           (Cap. 5)")
print("  05. Análisis de Comportamiento (UBA) (Cap. 6)")
print("  06. Explicabilidad (XAI)             (Cap. 7)")
print("  07. Ataques Adversariales            (Cap. 8)")
print("  08. Pipeline Integrado               (Cap. 9-10)")
print("=" * 60)
print("\n[OK] Pipeline de ciberseguridad con IA completado.")